# Inspect One Reasoning Chain

Use this notebook from `/gz-data/research/demo` after running `trajectory_signal_visualization.py`.

Change only `CHAIN_ID` or call `inspect_chain(66)` with another number to inspect a different chain. The notebook prints the problem, model response, parsed steps, gold first-error index, dynamic scores, and same-problem sibling rows when available.

In [ ]:
from pathlib import Path

NPZ_PATH = Path("data/features/full_gsm8k.npz")
VIZ_DIR = Path("outputs/trajectory_signal_visualization_full_gsm8k")
CHAIN_ID = 66
SIGNAL = "spread"
LAYER = 14
NEAREST_LAYER = True
MIN_PREFIX = 2

# Optional fallback when the npz does not store question text.
# It works only if the dataset is already cached or internet access is available.
TRY_HF_GSM8K = True
HF_DATASET = "openai/gsm8k"
HF_SUBSET = "main"
HF_SPLIT = "test"

In [ ]:
import json
import math
import re
import sys
import textwrap
from types import SimpleNamespace

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import Image, Markdown, display

def find_demo_dir():
    candidates = [Path.cwd(), Path("/gz-data/research/demo")]
    candidates.extend(Path.cwd().parents)
    for p in candidates:
        if (p / "trajectory_phase_transition_audit.py").exists():
            return p.resolve()
    raise FileNotFoundError(
        "Could not find trajectory_phase_transition_audit.py. "
        "Run this notebook from /gz-data/research/demo or set DEMO_DIR manually."
    )

DEMO_DIR = find_demo_dir()
if str(DEMO_DIR) not in sys.path:
    sys.path.insert(0, str(DEMO_DIR))
if not NPZ_PATH.is_absolute():
    NPZ_PATH = DEMO_DIR / NPZ_PATH
if not VIZ_DIR.is_absolute():
    VIZ_DIR = DEMO_DIR / VIZ_DIR
display(Markdown(f"Using demo dir: `{DEMO_DIR}`"))

import trajectory_phase_transition_audit as tpt

try:
    import trajectory_signal_visualization as tsv
except Exception:
    tsv = None

def _scalar(x):
    if isinstance(x, np.ndarray):
        if x.shape == ():
            return x.item()
        return x
    if isinstance(x, np.generic):
        return x.item()
    return x

def _text(x):
    x = _scalar(x)
    if x is None:
        return ""
    if isinstance(x, bytes):
        return x.decode("utf-8", errors="replace")
    if isinstance(x, (list, tuple, np.ndarray)):
        return "\n".join(str(_scalar(v)) for v in list(x))
    return str(x)

def _obj_at(arr, idx):
    val = arr[int(idx)]
    return _scalar(val)

def _field(data, names):
    for name in names:
        if name in data.files:
            return name
    return None

def _field_at(data, names, chain_id, *, by_problem=False):
    name = _field(data, names)
    if name is None:
        return None
    arr = data[name]
    idx = int(chain_id)
    if by_problem and "problem_ids" in data.files:
        pid = int(data["problem_ids"][idx])
        if len(arr) > pid:
            idx = pid
    if len(arr) <= idx:
        return None
    return _obj_at(arr, idx)

_HF_CACHE = None

def question_from_hf(problem_id):
    global _HF_CACHE
    if not TRY_HF_GSM8K:
        return ""
    try:
        from datasets import load_dataset
        if _HF_CACHE is None:
            _HF_CACHE = load_dataset(HF_DATASET, HF_SUBSET, split=HF_SPLIT)
        ex = _HF_CACHE[int(problem_id)]
        return _text(ex.get("question", ""))
    except Exception as e:
        return f"[question text unavailable; HF fallback failed: {type(e).__name__}: {e}]"

def get_question(data, chain_id):
    val = _field_at(
        data,
        ["questions", "question", "problem", "problems", "prompts", "prompt"],
        chain_id,
        by_problem=True,
    )
    if val is not None:
        return _text(val)
    if "problem_ids" in data.files:
        return question_from_hf(int(data["problem_ids"][int(chain_id)]))
    return "[question text unavailable: no question/prompt field in npz]"

def get_response(data, chain_id):
    val = _field_at(
        data,
        ["responses", "response", "generated_texts", "generations", "outputs", "texts"],
        chain_id,
    )
    return _text(val) if val is not None else "[response text unavailable: no responses field in npz]"

def get_steps(data, chain_id, response=""):
    val = _field_at(data, ["steps_text", "steps", "step_text"], chain_id)
    if val is not None:
        if isinstance(val, np.ndarray):
            return [_text(v).strip() for v in val.tolist()]
        if isinstance(val, (list, tuple)):
            return [_text(v).strip() for v in val]
        return [_text(val).strip()]
    lines = [ln.strip() for ln in str(response).splitlines() if ln.strip()]
    return lines

def get_label_info(data, chain_id):
    i = int(chain_id)
    out = {}
    for name in [
        "problem_ids", "sample_idx", "gold_error_step", "is_correct", "is_correct_strict",
        "format_ok", "pred_source", "pred_answers", "gold_answers", "n_steps",
    ]:
        if name in data.files and len(data[name]) > i:
            out[name] = _scalar(data[name][i])
    return out

def make_args():
    return SimpleNamespace(
        layer=LAYER,
        nearest_layer=NEAREST_LAYER,
        layer_is_id=False,
        max_chains=0,
        no_progress=True,
        min_prefix=MIN_PREFIX,
        scale_floor=1e-3,
        std_floor_frac=0.05,
        event_q=0.90,
        stable_q=0.50,
        topk=2,
        lse_tau=1.0,
        bootstrap=0,
        seed=13,
    )

In [ ]:
data = np.load(NPZ_PATH, allow_pickle=True)
args = make_args()
seqs, meta = tpt.get_signal_sequences(data, args)
seq_by_chain = {int(s["chain_idx"]): s for s in seqs}
event_rows = tpt.build_event_rows(seqs, SIGNAL, args)
chain_rows = tpt.build_chain_summaries(event_rows, SIGNAL, args)
if tsv is not None:
    chain_rows, dynamic_meta = tsv.augment_chain_rows(chain_rows, event_rows, args)
else:
    dynamic_meta = {}
chain_by_id = {int(r["chain_idx"]): r for r in chain_rows}
event_by_chain = {}
for r in event_rows:
    event_by_chain.setdefault(int(r.chain_idx), []).append(r)

display(Markdown(f"Loaded `{NPZ_PATH}`: {meta}"))
display(pd.DataFrame([dynamic_meta]) if dynamic_meta else Markdown("No dynamic meta from visualization helper."))

In [ ]:
def build_step_table(chain_id, signal=SIGNAL):
    chain_id = int(chain_id)
    seq = seq_by_chain[chain_id]
    row = chain_by_id.get(chain_id, {})
    response = get_response(data, chain_id)
    steps = get_steps(data, chain_id, response=response)
    T = int(seq["n_steps"])
    df = pd.DataFrame({"step": np.arange(T, dtype=int)})
    for name, arr in seq["signals"].items():
        x = np.asarray(arr, dtype=float)
        df[name] = x[:T]
    ev = {int(r.step_idx): r for r in event_by_chain.get(chain_id, [])}
    for name in ["phase", "signal_value", "level_z", "jump_z", "break_z", "shock_z", "prefix_mean", "prefix_std"]:
        vals = []
        for t in range(T):
            rr = ev.get(t)
            if rr is None:
                vals.append(np.nan if name != "phase" else "warmup")
            elif name == "phase":
                vals.append(rr.phase)
            else:
                vals.append(float(rr.features.get(name, np.nan)))
        df[name] = vals
    if "step_token_ranges" in data.files:
        rng = np.asarray(data["step_token_ranges"][chain_id], dtype=int)
        if rng.ndim == 2 and len(rng) >= T:
            df["token_lo"] = rng[:T, 0]
            df["token_hi"] = rng[:T, 1]
            df["n_tokens"] = rng[:T, 1] - rng[:T, 0] + 1
    step_text = [steps[t] if t < len(steps) else "" for t in range(T)]
    gold = int(row.get("gold_error_step", -1))
    df["is_gold_first_error"] = df["step"].eq(gold)
    df["step_text"] = step_text
    return df

def plot_chain(df, row, chain_id, signal=SIGNAL):
    gold = int(row.get("gold_error_step", -1))
    peak_break = int(row.get("argmax_break_z", -1))
    fig, ax = plt.subplots(2, 1, figsize=(11, 5.5), sharex=True)
    ax[0].plot(df["step"], df[signal], "-o", label=signal)
    ax[0].set_ylabel(signal)
    ax[0].legend(loc="upper left")
    ax[0].grid(alpha=0.25)
    for name, style in [("break_z", "-o"), ("level_z", "--"), ("jump_z", ":"), ("shock_z", "-.")]:
        if name in df:
            ax[1].plot(df["step"], df[name], style, label=name)
    ax[1].set_ylabel("prefix event score")
    ax[1].set_xlabel("step")
    ax[1].legend(loc="upper left", ncol=4)
    ax[1].grid(alpha=0.25)
    for a in ax:
        if gold >= 0:
            a.axvline(gold, color="red", lw=1.2, alpha=0.8, label="gold first error")
        if peak_break >= 0:
            a.axvline(peak_break, color="black", lw=0.9, ls="--", alpha=0.75, label="argmax break")
    title = f"chain {chain_id} | problem {row.get('problem_id')} | gold {gold} | error {row.get('y_chain_error')} | argmax_break {peak_break}"
    fig.suptitle(title)
    fig.tight_layout()
    plt.show()

def display_text_block(title, text, max_chars=None):
    body = str(text)
    if max_chars is not None and len(body) > max_chars:
        body = body[:max_chars] + "\n...[truncated]"
    display(Markdown(f"### {title}\n```text\n{body}\n```"))

def show_case_card(chain_id, signal=SIGNAL):
    p = VIZ_DIR / "cases" / f"{NPZ_PATH.stem}_{signal}_chain_{int(chain_id):05d}.png"
    if p.exists():
        display(Markdown(f"### Existing case card: `{p}`"))
        display(Image(filename=str(p)))
    else:
        display(Markdown(f"No case card found at `{p}`. Re-run visualization with larger `--case_count` or `--plot_all_cases`."))

def show_same_problem_siblings(problem_id, signal=SIGNAL, limit=30):
    csv_path = VIZ_DIR / f"{NPZ_PATH.stem}_{signal}_chain_dynamic_scores.csv"
    if not csv_path.exists():
        display(Markdown(f"No chain CSV found at `{csv_path}`."))
        return
    cdf = pd.read_csv(csv_path)
    same = cdf[cdf["problem_id"].astype(int).eq(int(problem_id))].copy()
    cols = [
        "chain_idx", "problem_id", "y_chain_error", "gold_error_step", "first_error_mode",
        "max_signal_value", "mean_signal_value", "max_break_z", "argmax_break_z",
        "gold_break_z_rank", "gold_break_z_percentile",
    ]
    cols = [c for c in cols if c in same.columns]
    display(Markdown(f"### Same-problem sibling chains: problem `{problem_id}`"))
    display(same[cols].sort_values(["y_chain_error", "max_signal_value"], ascending=[False, False]).head(limit))

def inspect_chain(chain_id=CHAIN_ID, signal=SIGNAL, *, show_response=True, show_card=True, max_response_chars=None):
    chain_id = int(chain_id)
    if chain_id not in seq_by_chain:
        raise KeyError(f"chain {chain_id} not loaded; available range starts {sorted(seq_by_chain)[:5]}")
    row = chain_by_id.get(chain_id, {})
    info = get_label_info(data, chain_id)
    problem_id = int(row.get("problem_id", info.get("problem_ids", -1)))
    question = get_question(data, chain_id)
    response = get_response(data, chain_id)
    df = build_step_table(chain_id, signal=signal)

    display(Markdown(f"## Chain `{chain_id}` | Problem `{problem_id}`"))
    display(pd.DataFrame([{**info, **row}]).T.rename(columns={0: "value"}))
    display_text_block("Problem", question)
    if show_response:
        display_text_block("Model Response", response, max_chars=max_response_chars)
    display(Markdown("### Parsed Steps and Signals"))
    display(df)
    plot_chain(df, row, chain_id, signal=signal)
    show_same_problem_siblings(problem_id, signal=signal)
    if show_card:
        show_case_card(chain_id, signal=signal)
    return df

In [ ]:
# Change this number to inspect a different chain.
df66 = inspect_chain(66, signal="spread", max_response_chars=None)

Useful follow-up calls:

```python
inspect_chain(12)
inspect_chain(66, signal="entropy_mean")
show_same_problem_siblings(problem_id=66)
```

Interpretation reminder: if `argmax_break_z` is far before `gold_error_step`, the signal is likely detecting a task-stage or core-computation transition rather than the actual first wrong step.